# Stage 9 — SHAP explainability for M0 tree models

P95 is the primary explanation. P90 is a sensitivity comparison. SHAP explains associations learned by the fitted tree models; it does not establish causality. Local TP/TN/FP/FN cases use each tree model's raw 0.5 decision rule, so they are explanations of the tree being interpreted rather than a different selected model.

In [ ]:
from pathlib import Path
import json, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, joblib
from IPython.display import display
from scipy.stats import spearmanr

warnings.filterwarnings("ignore",category=FutureWarning)
sns.set_theme(style="whitegrid")
ROOT=Path.cwd().resolve()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"/"processed"; MODEL_DIR=ROOT/"models"
TABLE_DIR=ROOT/"reports"/"tables"/"09_shap_analysis"; TABLE_DIR.mkdir(parents=True,exist_ok=True)
FIGURE_DIR=ROOT/"reports"/"figures"/"09_shap_analysis"; FIGURE_DIR.mkdir(parents=True,exist_ok=True)
RANDOM_STATE=42; SHAP_SAMPLE_SIZE=2000; BOOTSTRAP_REPETITIONS=100
COMMON_FEATURES=["lag1_peak_kw","lag1_mean_kw","lag1_energy_kwh","lag1_coverage_ratio","lag1_was_complete","peak_mean_7d","peak_mean_30d","peak_max_7d","peak_max_30d","peak_std_7d","peak_std_30d","energy_mean_7d","energy_mean_30d","lag1_peak_over_p95","lag1_peak_minus_p95","peak_mean_7d_over_30d","lag1_peak_zscore_30d","day_of_week_sin","day_of_week_cos","is_weekend","month_sin","month_cos"]
EXPERIMENTS={"p95":{"path":DATA/"05_model_ready_m0_p95.parquet","role":"primary","features":COMMON_FEATURES+["p95_high_count_7d","p95_high_count_30d","p95_days_since_last_high"]},"p90":{"path":DATA/"05_model_ready_m0_p90.parquet","role":"sensitivity","features":COMMON_FEATURES+["p90_high_count_7d","p90_high_count_30d","p90_days_since_last_high"]}}

def save_table(df,name,title=None):
    df.to_csv(TABLE_DIR/name,index=False)
    if title: print(title)
    display(df)

def positive_class_explanation(model,transformed,feature_names):
    raw=shap.TreeExplainer(model)(transformed)
    values=np.asarray(raw.values); base=np.asarray(raw.base_values)
    if values.ndim==3:
        values=values[:,:,1]
        if base.ndim==2: base=base[:,1]
    if base.ndim==0: base=np.repeat(base,len(transformed))
    return shap.Explanation(values=values,base_values=base,data=np.asarray(transformed),feature_names=list(feature_names))

def bootstrap_rank_stability(values,repetitions=100,seed=42):
    rng=np.random.default_rng(seed); full=np.abs(values).mean(axis=0); full_rank=pd.Series(full).rank()
    scores=[]
    for _ in range(repetitions):
        idx=rng.integers(0,len(values),len(values)); current=pd.Series(np.abs(values[idx]).mean(axis=0)).rank(); scores.append(spearmanr(full_rank,current).statistic)
    return float(np.nanmedian(scores)),float(np.nanpercentile(scores,2.5)),float(np.nanpercentile(scores,97.5))
print("SHAP version",shap.__version__)

## 9.1 Validate the saved tree models and deterministic explanation samples

In [ ]:
artifacts={}; sample_rows=[]; validation=[]
for key,spec in EXPERIMENTS.items():
    df=pd.read_parquet(spec["path"]); df["measurement_date"]=pd.to_datetime(df["measurement_date"])
    test=df.loc[df.split_name.eq("test")].sort_values(["measurement_date","household_id"]).copy()
    model=joblib.load(MODEL_DIR/f"07_{key}_best_tree_model_m0.joblib")
    imputer=model.named_steps["imputer"]; tree=model.named_steps["model"]
    family="xgboost" if tree.__class__.__name__.lower().startswith("xgb") else "random_forest"
    sample=test.sample(n=min(SHAP_SAMPLE_SIZE,len(test)),random_state=RANDOM_STATE).sort_index()
    transformed=imputer.transform(sample[spec["features"]]); names=imputer.get_feature_names_out(spec["features"])
    explanation=positive_class_explanation(tree,transformed,names)
    artifacts[key]={"df":df,"test":test,"pipeline":model,"tree":tree,"family":family,"sample":sample,"transformed":transformed,"names":names,"explanation":explanation}
    sample_rows.append({"target":key.upper(),"experiment_role":spec["role"],"tree_family":family,"test_rows":len(test),"sample_rows":len(sample),"sample_events":int(sample.target.sum()),"transformed_features":len(names)})
    validation.extend([(key,"tree_family",family in {"random_forest","xgboost"}),(key,"sample_deterministic",sample.index.equals(test.sample(n=min(SHAP_SAMPLE_SIZE,len(test)),random_state=RANDOM_STATE).sort_index().index)),(key,"finite_shap",np.isfinite(explanation.values).all()),(key,"shape_matches",explanation.values.shape==transformed.shape)])
samples=pd.DataFrame(sample_rows); save_table(samples,"09_01_explanation_samples.csv","Deterministic SHAP samples")
input_checks=pd.DataFrame(validation,columns=["target","check","passed"]); save_table(input_checks,"09_02_input_validation.csv","SHAP input validation"); assert input_checks.passed.all()

## 9.2 Global P95 explanation (primary) and P90 sensitivity explanation

In [ ]:
importance_tables=[]; stability_rows=[]
for key,a in artifacts.items():
    importance=pd.DataFrame({"feature":a["names"],"mean_abs_shap":np.abs(a["explanation"].values).mean(axis=0),"mean_shap":a["explanation"].values.mean(axis=0)}).sort_values("mean_abs_shap",ascending=False)
    importance.insert(0,"target",key.upper()); importance["rank"]=np.arange(1,len(importance)+1); importance_tables.append(importance)
    median,low,high=bootstrap_rank_stability(a["explanation"].values,BOOTSTRAP_REPETITIONS,RANDOM_STATE)
    stability_rows.append({"target":key.upper(),"tree_family":a["family"],"bootstrap_repetitions":BOOTSTRAP_REPETITIONS,"median_spearman_rank_stability":median,"ci_2_5":low,"ci_97_5":high})
    plt.figure(figsize=(10,7)); shap.plots.beeswarm(a["explanation"],max_display=15,show=False); plt.title(f"{key.upper()} M0 global SHAP summary ({EXPERIMENTS[key]['role']})"); plt.tight_layout(); plt.savefig(FIGURE_DIR/f"09_{1 if key=='p95' else 3:02d}_{key}_global_beeswarm.png",dpi=180,bbox_inches="tight"); plt.show()
    top=importance.head(15).sort_values("mean_abs_shap")
    fig,ax=plt.subplots(figsize=(9,6)); ax.barh(top.feature,top.mean_abs_shap,color="#b2182b" if key=="p95" else "#2166ac"); ax.set(title=f"{key.upper()} mean absolute SHAP importance",xlabel="Mean |SHAP value|",ylabel=""); plt.tight_layout(); plt.savefig(FIGURE_DIR/f"09_{2 if key=='p95' else 4:02d}_{key}_global_importance.png",dpi=180,bbox_inches="tight"); plt.show()
importance_all=pd.concat(importance_tables,ignore_index=True); save_table(importance_all,"09_03_global_shap_importance.csv","Global mean absolute SHAP importance")
stability=pd.DataFrame(stability_rows); save_table(stability,"09_04_bootstrap_rank_stability.csv","Bootstrap feature-rank stability")

## 9.3 Dependence views for leading features

In [ ]:
dependence_rows=[]
for key,a in artifacts.items():
    top_features=importance_all.loc[importance_all.target.eq(key.upper())].nsmallest(2,"rank").feature.tolist()
    fig,axes=plt.subplots(1,2,figsize=(12,4.8))
    for ax,feature in zip(axes,top_features):
        j=list(a["names"]).index(feature); x=np.asarray(a["transformed"])[:,j]; y=a["explanation"].values[:,j]
        hb=ax.hexbin(x,y,gridsize=35,mincnt=1,cmap="viridis"); ax.axhline(0,color="grey",lw=.8); ax.set(title=feature,xlabel="Imputed feature value",ylabel="SHAP contribution"); fig.colorbar(hb,ax=ax,label="Sample density")
        dependence_rows.append({"target":key.upper(),"feature":feature,"spearman_value_vs_shap":spearmanr(x,y).statistic})
    fig.suptitle(f"{key.upper()} leading-feature dependence views"); plt.tight_layout(); plt.savefig(FIGURE_DIR/f"09_{5 if key=='p95' else 6:02d}_{key}_dependence.png",dpi=180,bbox_inches="tight"); plt.show()
save_table(pd.DataFrame(dependence_rows),"09_05_dependence_summary.csv","Leading-feature dependence summaries")

## 9.4 Local P95 TP, TN, FP and FN explanations

Cases are defined using the best P95 tree model's own uncalibrated 0.5 rule. This avoids attributing decisions from the separately selected logistic model to a random forest explanation.

In [ ]:
key="p95"; a=artifacts[key]; test=a["test"].copy(); raw=a["pipeline"].predict_proba(test[EXPERIMENTS[key]["features"]])[:,1]; pred=(raw>=.5).astype(int); y=test.target.astype(int).to_numpy()
case_masks={"TP":(y==1)&(pred==1),"TN":(y==0)&(pred==0),"FP":(y==0)&(pred==1),"FN":(y==1)&(pred==0)}
case_indices={"TP":np.flatnonzero(case_masks["TP"])[np.argmax(raw[case_masks["TP"]])],"TN":np.flatnonzero(case_masks["TN"])[np.argmin(raw[case_masks["TN"]])],"FP":np.flatnonzero(case_masks["FP"])[np.argmax(raw[case_masks["FP"]])],"FN":np.flatnonzero(case_masks["FN"])[np.argmin(raw[case_masks["FN"]])]}
local=test.iloc[list(case_indices.values())].copy(); local["case_type"]=list(case_indices.keys()); local["tree_raw_probability"]=[raw[i] for i in case_indices.values()]; local["tree_prediction"]=[pred[i] for i in case_indices.values()]
local_transformed=a["pipeline"].named_steps["imputer"].transform(local[EXPERIMENTS[key]["features"]]); local_exp=positive_class_explanation(a["tree"],local_transformed,a["names"])
case_table=local[["case_type","row_id","household_id","measurement_date","target","tree_prediction","tree_raw_probability"]]
save_table(case_table,"09_06_p95_local_cases.csv","P95 local explanation cases")
for i,case in enumerate(local.case_type):
    plt.figure(figsize=(9,6)); shap.plots.waterfall(local_exp[i],max_display=12,show=False); plt.title(f"P95 tree {case}: row {local.iloc[i].row_id}"); plt.tight_layout(); plt.savefig(FIGURE_DIR/f"09_07_p95_local_{case.lower()}.png",dpi=180,bbox_inches="tight"); plt.show()

## 9.5 P95–P90 driver comparison

In [ ]:
p95_imp=importance_all.loc[importance_all.target.eq("P95"),["feature","mean_abs_shap","rank"]].rename(columns={"mean_abs_shap":"p95_mean_abs_shap","rank":"p95_rank"})
p90_imp=importance_all.loc[importance_all.target.eq("P90"),["feature","mean_abs_shap","rank"]].rename(columns={"mean_abs_shap":"p90_mean_abs_shap","rank":"p90_rank"})
comparison=p95_imp.merge(p90_imp,on="feature",how="outer")
save_table(comparison.sort_values("p95_rank",na_position="last"),"09_07_p95_p90_importance_comparison.csv","P95/P90 driver comparison")
common=comparison.dropna(); rank_correlation=spearmanr(common.p95_rank,common.p90_rank).statistic if len(common)>1 else np.nan
top10_p95=set(p95_imp.nsmallest(10,"p95_rank").feature); top10_p90=set(p90_imp.nsmallest(10,"p90_rank").feature)
comparison_summary=pd.DataFrame([{"common_features_compared":len(common),"rank_spearman":rank_correlation,"top10_overlap_count":len(top10_p95&top10_p90),"top10_jaccard":len(top10_p95&top10_p90)/len(top10_p95|top10_p90),"p95_rank_stability":stability.loc[stability.target.eq("P95"),"median_spearman_rank_stability"].iloc[0],"p90_rank_stability":stability.loc[stability.target.eq("P90"),"median_spearman_rank_stability"].iloc[0],"p90_more_stable_on_this_sample":stability.loc[stability.target.eq("P90"),"median_spearman_rank_stability"].iloc[0]>stability.loc[stability.target.eq("P95"),"median_spearman_rank_stability"].iloc[0]}])
save_table(comparison_summary,"09_08_sensitivity_comparison_summary.csv","Sensitivity comparison summary")

plot=comparison.dropna().nlargest(15,"p95_mean_abs_shap").sort_values("p95_mean_abs_shap")
fig,ax=plt.subplots(figsize=(10,7)); yloc=np.arange(len(plot)); ax.barh(yloc-.18,plot.p95_mean_abs_shap,height=.35,label="P95 primary",color="#b2182b"); ax.barh(yloc+.18,plot.p90_mean_abs_shap,height=.35,label="P90 sensitivity",color="#2166ac"); ax.set_yticks(yloc,plot.feature); ax.set(title="Common-driver SHAP magnitude comparison",xlabel="Mean |SHAP value|",ylabel=""); ax.legend(); plt.tight_layout(); plt.savefig(FIGURE_DIR/"09_08_p95_p90_driver_comparison.png",dpi=180,bbox_inches="tight"); plt.show()

## 9.6 Final controls and manifest

In [ ]:
checks=pd.DataFrame([
 ("p95_primary_outputs",all((FIGURE_DIR/f).exists() for f in ["09_01_p95_global_beeswarm.png","09_02_p95_global_importance.png","09_05_p95_dependence.png"])),
 ("p90_sensitivity_outputs",all((FIGURE_DIR/f).exists() for f in ["09_03_p90_global_beeswarm.png","09_04_p90_global_importance.png","09_06_p90_dependence.png"])),
 ("four_p95_local_cases",len(case_table)==4 and set(case_table.case_type)=={"TP","TN","FP","FN"}),
 ("all_shap_finite",all(np.isfinite(a["explanation"].values).all() for a in artifacts.values())),
 ("deterministic_sample_size",all(len(a["sample"])==SHAP_SAMPLE_SIZE for a in artifacts.values())),
 ("noncausal_scope_recorded",True),
],columns=["check","passed"])
save_table(checks,"09_09_final_validation.csv","Stage 9 final validation"); assert checks.passed.all(),checks.loc[~checks.passed]
manifest={"stage":"09_shap_analysis","primary_target":"P95","sensitivity_target":"P90","explained_models":{k:a["family"] for k,a in artifacts.items()},"sample_size":SHAP_SAMPLE_SIZE,"random_state":RANDOM_STATE,"interpretation":"associational model explanations, not causal effects","comparison":comparison_summary.to_dict("records")[0],"software":{"python":platform.python_version(),"pandas":pd.__version__,"shap":shap.__version__}}
(TABLE_DIR/"09_run_manifest.json").write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
print("Stage 9 complete. P95 remains the principal explanation; P90 is a sensitivity comparison.")

## Interpretation boundary

Large absolute SHAP values identify features the fitted model relied on for these predictions. They do not show that changing a feature would cause demand risk to change. Correlated lag/rolling variables can share or redistribute importance, and the explanation applies only to the M0 data, split, population, and fitted tree family used here.